# Matrix storage

In [ ]:
#    APM41012EP course notebook - Chapter 4 - M. Massot 2026-2027 - École polytechnique
#    ----------   
#    Matrix storage
#    
#    Authors: L. Séries and M. Massot - (C) 2026

## Dense storage

Naturally, two-dimensional arrays are well suited to storing matrices. For example, we can use a two-dimensional `numpy` array to store the matrix:

$$
A = \begin{bmatrix}
1 & 0 & 0 & 3 \\
2 & 5 & 0 & 7 \\
0 & 4 & 2 & 0 \\
5 & 0 & 0 & 1
\end{bmatrix}
$$

In [ ]:
import numpy as np
A = np.array([[1,0,0,3], [2,5,0,7], [0,4,2,0], [5,0,0,1]])
print(A)

A square matrix of size $(n \times n)$ whose coefficients are stored as 64-bit floats occupies $8 \times (n \times n)$ bytes in memory. 

Examples:
* a matrix of size $(1024 \times 1024)$ occupies 8 MiB (discretisation matrix of the Laplacian on a 1d grid with 1024 subdivisions) 
* a matrix of size $(2048 \times 2048)$ occupies 32 MiB
* a matrix of size $(65536 \times 65536)$ occupies 32 GiB
* a matrix of size $(1048576 \times 1048576)$ occupies 8 TiB (discretisation matrix of the Laplacian on a 2d grid with 1024x1024 subdivisions) 

The `nbytes` function returns the memory usage in bytes 

In [ ]:
n = 2048
Rand = np.random.rand(n,n)
print(f"Size of the ({n}x{n}) matrix: {Rand.nbytes} bytes = {Rand.nbytes/(1024*1024)} MiB")

Matrices arising from the spatial discretisation of partial differential equations mostly contain zeros.

To store these matrices, we do not use two-dimensional arrays but data structures suited to the sparse nature of these matrices.

## Sparse matrix: COO (Coordinate Format)

The COO data structure can be represented by:

* a `data` array containing the non-zero coefficients of the matrix (in any order)
* a `row` array containing the row indices of each element of `data`
* a `col` array containing the column indices of each element of `data`

Example for the matrix:

$$
A = \begin{bmatrix}
1 & 0 & 0 & 3 \\
2 & 5 & 0 & 7 \\
0 & 4 & 2 & 0 \\
5 & 0 & 0 & 1
\end{bmatrix}
$$

In [ ]:
from scipy.sparse import coo_matrix

data = np.array([1, 3, 2, 5, 4, 5, 1, 2, 7])

row = np.array([0, 0, 1, 1, 2, 3, 3, 2, 1])
col = np.array([0, 3, 0, 1, 1, 0, 3, 2, 3])

A_coo = coo_matrix((data, (row, col)), shape=(4, 4))
A_coo.todense()

This is the format used by the sparse linear system solver library [Mumps](http://mumps.enseeiht.fr/).

To store the finite-difference discretisation matrix of the Laplacian on a 2d grid with 1024x1024 subdivisions, 5 non-zero elements per row must be stored, i.e. for a COO sparse matrix:
* $ 5 \times 1024 \times 1024 = 41943040$ double-precision floats (8 bytes), i.e. 40 MiB for the `data` array 
* $ 5 \times 1024 \times 1024 = 41943040$ integers (4 bytes), i.e. 20 MiB for the `row` array 
* $ 5 \times 1024 \times 1024 = 41943040$ integers (4 bytes), i.e. 20 MiB for the `col` array 

which occupies 80 MiB instead of 8 TiB for dense storage!

## Sparse matrix: CSR (Compressed Sparse Row)

The CSR data structure can be represented by:

* a `data` array containing the non-zero coefficients of the matrix arranged row by row
* a `col` array containing the column indices of each element of `data`
* a `row_ptr` array whose element `i` contains the index in the `data` and `col` arrays of the first non-zero entry of row `i` of the matrix

Example for the matrix:

$$
A = \begin{bmatrix}
1 & 0 & 0 & 3 \\
2 & 5 & 0 & 7 \\
0 & 4 & 2 & 0 \\
5 & 0 & 0 & 1
\end{bmatrix}
$$

In [ ]:
from scipy.sparse import csr_matrix

data = np.array([1, 3, 2, 5, 7, 4, 2, 5, 1])

col = [0, 3, 0, 1, 3, 1, 2, 0, 3]
row_ptr = [0, 2, 5, 7, 9]

A_csr = csr_matrix((data, col, row_ptr), shape=(4, 4))
A_csr.todense()

To store the finite-difference discretisation matrix of the Laplacian on a 2d grid with 1024x1024 subdivisions, 5 non-zero elements per row must be stored, i.e. for a CSR sparse matrix:
* $ 5 \times 1024 \times 1024 = 41943040$ double-precision floats (8 bytes), i.e. 40 MiB for the `data` array 
* $ 5 \times 1024 \times 1024 = 41943040$ integers (4 bytes), i.e. 20 MiB for the `col` array 
* $ 1024 \times 1024 + 1 = 1048577 $ integers (4 bytes), i.e. 4 MiB for the `row_ptr` array 

which occupies 64 MiB instead of 8 TiB for dense storage!

## Sparse matrix: CSC (Compressed Sparse Column)

The CSC data structure can be represented by:

* a `data` array containing the non-zero coefficients of the matrix arranged column by column
* a `row` array containing the row indices of each element of `data`
* a `col_ptr` array whose element `i` contains the index in the `data` and `row` arrays of the first non-zero entry of column `i` of the matrix

This is the format used by the sparse linear system solver library [SuperLU](https://portal.nersc.gov/project/sparse/superlu/).



Example:

$$
A = \begin{bmatrix}
1 & 0 & 0 & 3 \\
2 & 5 & 0 & 7 \\
0 & 4 & 2 & 0 \\
5 & 0 & 0 & 1
\end{bmatrix}
$$

In [ ]:
from scipy.sparse import csc_matrix

data = np.array([1, 2, 5, 5, 4, 2, 3, 7, 1])

row = [0, 1, 3, 1, 2, 2, 0, 1, 3]
col_ptr = [0, 3, 5, 6, 9]

A_csc = csc_matrix((data, row, col_ptr), shape=(4, 4))
A_csc.todense()

The memory occupied by the storage of the finite-difference discretisation matrix of the Laplacian on a 2d grid with 1024x1024 subdivisions for a CSC matrix is the same as for the CSR structure, since the Laplacian matrix is symmetric.

## Sparse matrix: diagonal storage

The diagonal storage data structure can be represented by:

* a two-dimensional `diags` array ab containing the coefficients of each diagonal of the matrix
* an integer array `offset` containing the position of each diagonal relative to the main diagonal

Remarks: this data structure can store zero elements.

Example:

$$
A = \begin{bmatrix}
5 & 2 & 6  & 0  & 0  \\
1 & 4 & 2  & 5  & 0  \\
0 & 1 & 3  & 2  & 1  \\
0 & 0 & 1  & 2  & 0  \\
0 & 0 & 0  & 1  & 1
\end{bmatrix}
$$

In [ ]:
from scipy.sparse import dia_matrix

data = np.array([[5, 4, 3, 2, 1], [1, 1, 1, 1, 0], [0, 2, 2, 2, 0], [0, 0, 6, 5, 1]])

offset = np.array([0, -1, 1, 2])

A_dia = dia_matrix((data, offset), shape=(5, 5))
A_dia.todense()

## References 

* [Sparse Matrix Storage Formats, *Jack Dongarra*](http://www.netlib.org/utk/people/JackDongarra/etemplates/node372.html)
* [Scipy sparse matrix classes](https://docs.scipy.org/doc/scipy/reference/sparse.html)